In [2]:
import pandas as pd
import numpy as np
df=pd.read_csv('new_cleaned_exoplanets.csv')

# **Filling the most can missing values**

In [3]:
# Convert the orbital period of Earth into numerical value.
df['pl_orb_period(in E.years)'] = pd.to_numeric(df['pl_orb_period(in E.years)'], errors='coerce')
df['pl_orb_period(in E.years)'] = df['pl_orb_period(in E.years)'].replace(0, np.nan)

df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 6128 entries, 0 to 6127
Data columns (total 20 columns):
 #   Column                                   Non-Null Count  Dtype  
---  ------                                   --------------  -----  
 0   pl_name                                  6128 non-null   object 
 1   Star's name                              6128 non-null   object 
 2   sy_snum                                  6128 non-null   int64  
 3   sy_pnum                                  6128 non-null   int64  
 4   discoverymethod                          6128 non-null   object 
 5   disc_year                                6127 non-null   float64
 6   pl_orb_period(in days)                   5776 non-null   float64
 7   pl_orb_period(in E.years)                3841 non-null   float64
 8   pl_orb_semi_maj_axis(in AU)              3821 non-null   float64
 9   pl_orb_eccent                            2586 non-null   float64
 10  pl_rad(in Earth Radius)                  4553 no

In [4]:
# Filling the missing values of the masses and radiuses.
df['st_mass(in solar mass)'] = df['st_mass(in solar mass)'].fillna(df['st_rad(in solar radius)']**(1/0.8))
df['st_rad(in solar radius)'] = df['st_rad(in solar radius)'].fillna(df['st_mass(in solar mass)']**0.8)

df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 6128 entries, 0 to 6127
Data columns (total 20 columns):
 #   Column                                   Non-Null Count  Dtype  
---  ------                                   --------------  -----  
 0   pl_name                                  6128 non-null   object 
 1   Star's name                              6128 non-null   object 
 2   sy_snum                                  6128 non-null   int64  
 3   sy_pnum                                  6128 non-null   int64  
 4   discoverymethod                          6128 non-null   object 
 5   disc_year                                6127 non-null   float64
 6   pl_orb_period(in days)                   5776 non-null   float64
 7   pl_orb_period(in E.years)                3841 non-null   float64
 8   pl_orb_semi_maj_axis(in AU)              3821 non-null   float64
 9   pl_orb_eccent                            2586 non-null   float64
 10  pl_rad(in Earth Radius)                  4553 no

In [5]:
# The orbital period and the semi major axis

calc_a = (df['st_mass(in solar mass)'] * (df['pl_orb_period(in days)'] / 365.25)**2)**(1/3)
df['pl_orb_semi_maj_axis(in AU)'] = df['pl_orb_semi_maj_axis(in AU)'].fillna(calc_a)

calc_p = 365.25 * np.sqrt((df['pl_orb_semi_maj_axis(in AU)']**3) / df['st_mass(in solar mass)'])
df['pl_orb_period(in days)'] = df['pl_orb_period(in days)'].fillna(calc_p)

df['pl_orb_period(in E.years)'] = df['pl_orb_period(in days)'] / 365.25

In [6]:
# Filling the Eccentricity randomly [0,0.1] with a<1.1
cond = df['pl_orb_eccent'].isnull() & (df['pl_orb_semi_maj_axis(in AU)'] < 1.15)

num_to_fill = cond.sum()
random_ecc = np.random.uniform(0.0, 0.1, size=num_to_fill)

df.loc[cond, 'pl_orb_eccent'] = random_ecc

In [7]:
# Rocky planets where R<2
condition_rocky_R = (df['pl_rad(in Earth Radius)'] < 2) & df['pl_bmass(in Earth Mass)'].isnull()
df.loc[condition_rocky_R, 'pl_bmass(in Earth Mass)'] = df.loc[condition_rocky_R, 'pl_rad(in Earth Radius)']**3.68

condition_gas_R = (df['pl_rad(in Earth Radius)'] >= 2) & df['pl_bmass(in Earth Mass)'].isnull()
df.loc[condition_gas_R, 'pl_bmass(in Earth Mass)'] = 2.69 * (df.loc[condition_gas_R, 'pl_rad(in Earth Radius)']**0.93)

# Gas Planets where R>2
condition_rocky_M = (df['pl_bmass(in Earth Mass)'] < 13) & df['pl_rad(in Earth Radius)'].isnull()
df.loc[condition_rocky_M, 'pl_rad(in Earth Radius)'] = df.loc[condition_rocky_M, 'pl_bmass(in Earth Mass)']**(1/3.68)

condition_gas_M = (df['pl_bmass(in Earth Mass)'] >= 13) & df['pl_rad(in Earth Radius)'].isnull()
df.loc[condition_gas_M, 'pl_rad(in Earth Radius)'] = (df.loc[condition_gas_M, 'pl_bmass(in Earth Mass)'] / 2.69)**(1/0.93)

In [8]:
# Recalculate The physical properties

df['pl_dens(in (Earth mass/Earth radius))'] = 5.51 * (df['pl_bmass(in Earth Mass)'] / (df['pl_rad(in Earth Radius)']**3))
df['st_lum(in Solar luminosity)'] = (df['st_rad(in solar radius)']**2) * ((df['st_teff(in kelvin)'] / 5778)**4)
df['pl_recei_flux(in Earth\'s received flux)'] = df['st_lum(in Solar luminosity)'] / (df['pl_orb_semi_maj_axis(in AU)']**2)
df['pl_teff(in kelvin)'] = df['st_teff(in kelvin)'] * np.sqrt((df['st_rad(in solar radius)'] * 0.00465) / (2 * df['pl_orb_semi_maj_axis(in AU)']))

# Drop null values

In [9]:
# Specify the colomns that we remove the missing values
df = df.dropna(subset=['st_mass(in solar mass)', 'st_rad(in solar radius)', 'st_teff(in kelvin)'])
df = df.dropna(subset=['pl_bmass(in Earth Mass)', 'pl_rad(in Earth Radius)'], how='all')

df.info()

<class 'pandas.core.frame.DataFrame'>
Index: 5394 entries, 0 to 6127
Data columns (total 20 columns):
 #   Column                                   Non-Null Count  Dtype  
---  ------                                   --------------  -----  
 0   pl_name                                  5394 non-null   object 
 1   Star's name                              5394 non-null   object 
 2   sy_snum                                  5394 non-null   int64  
 3   sy_pnum                                  5394 non-null   int64  
 4   discoverymethod                          5394 non-null   object 
 5   disc_year                                5393 non-null   float64
 6   pl_orb_period(in days)                   5393 non-null   float64
 7   pl_orb_period(in E.years)                5393 non-null   float64
 8   pl_orb_semi_maj_axis(in AU)              5393 non-null   float64
 9   pl_orb_eccent                            5342 non-null   float64
 10  pl_rad(in Earth Radius)                  5394 non-nul

In [10]:
# Calling more data sources for adding new parameter, which is the parallax of the system.
df_new = pd.read_csv("new_parallax.csv")

In [11]:
df_new.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 37067 entries, 0 to 37066
Data columns (total 2 columns):
 #   Column   Non-Null Count  Dtype  
---  ------   --------------  -----  
 0   pl_name  37067 non-null  object 
 1   sy_plx   37067 non-null  float64
dtypes: float64(1), object(1)
memory usage: 579.3+ KB


In [12]:
df_new.columns

Index(['pl_name', 'sy_plx'], dtype='object')

In [13]:
# left join or left merge

df_merged = pd.merge(df, df_new[['pl_name', 'sy_plx']], on='pl_name', how='left')


df_merged['sy_plx'] = pd.to_numeric(df_merged['sy_plx'], errors='coerce')
valid_parallax = df_merged['sy_plx'] > 0
distance = 1000 / df_merged.loc[valid_parallax, 'sy_plx']

# Filling the missing data using the distance law
df_merged['sy_dist(in Parsec)'] = df_merged['sy_dist(in Parsec)'].fillna(distance)

output_file = "Data_Cleaned_Silver_layer.csv"
df_merged.to_csv(output_file, index=False)